In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set(style='ticks')
import loss_profit
from datetime import datetime

In [18]:
# this is wrapper for transaction function
# model = 2-class regression
# commission = 5 USD
# our_strategy vs buy_and_hold

def make_transactions(stock_names):
    
    if type(stock_names) is not list:
        stock_names = [stock_names]
#     print("Preparing adjusted close dataframe...")
    prices = loss_profit.prepare_adj_close(stock_names)

#     print("Calculating final capital using prediction model...")
    capital, shares, record_transactions_df, record_shares_df = loss_profit.buy_sell_regr(
        stock_names = stock_names, 
        predictions_name = 'predictions_model_regr_100epoch_qratio_0_2017_07_16 20_25_16_566179', 
        adj_close = prices, 
        buy_thr=.0, 
        sell_thr=-.0, 
        transaction_cost=5,verbose=False)
    #capital, shares = loss_profit.buy_sell_class3(predictions_name = 'predictions_model_class_100epoch_2017_07_11 17_08_21_002137', adj_close = prices, transaction_cost=5)
    #capital, shares = loss_profit.buy_sell_class2(predictions_name = 'predictions_model_class_100epoch_2017_07_13 13_15_09_200788', adj_close = prices, transaction_cost=5)
   
#     print("Final captial:")
#     print(capital)
#     print("Final shares:")
#     print(shares)
    
    return record_transactions_df, record_shares_df

def buy_and_hold_transaction(stock_names):
    if type(stock_names) is not list:
        stock_names = [stock_names]
    prices = loss_profit.prepare_adj_close(stock_names)
    buy_hold_final_capital, buy_hold_final_shares = loss_profit.buy_hold(stock_names, prices,verbose=False)
    return buy_hold_final_capital, buy_hold_final_shares
    

In [19]:
# create transaction dataframe
def create_transaction_dataframe(record_buy_sell_df,record_shares_df,stock_names):
    if type(stock_names) is not list:
        stock_names = [stock_names]
    return_df = pd.DataFrame()
    for stock_name in stock_names:
        buy_sell_df = record_buy_sell_df.loc[record_buy_sell_df.Names == stock_name]
        
        trans = pd.DataFrame()
        buy_df = buy_sell_df.loc[buy_sell_df.Operations == 'buy'].reset_index(drop=True)
        sell_df = buy_sell_df.loc[buy_sell_df.Operations == 'sell'].reset_index(drop=True)

        trans['sell_date'] = sell_df['Dates'].copy()
        trans['buy_date'] = buy_df['Dates'].copy()
        
        c_df = create_capital_dataframe(stock_names, record_shares_df, record_buy_sell_df)
        trans['capital_before'] = trans.buy_date.apply(lambda x: c_df.capital.loc[c_df.date == x].values[0] )
        trans['capital_after'] = trans.sell_date.apply(lambda x: c_df.capital.loc[c_df.date == x].values[0] )

        trans['return'] = (trans.capital_after - trans.capital_before)/trans.capital_before
        trans['t_period'] = trans.apply(lambda x: day_between(x.sell_date,x.buy_date).days, axis=1)
        trans['name'] = [stock_name for i in range(trans.shape[0])]
        return_df = pd.concat([return_df,trans])
    return_df = return_df.sort_values(['sell_date'],axis=0).reset_index(drop=True)
    return return_df

In [20]:
def get_amount_by_name_and_date(record_shares_df, name, date):
    return record_shares_df[name].loc[record_shares_df.date == date].values[0]

def create_capital_dataframe(stock_names,record_shares_df,record_buy_sell_df):
    if type(stock_names) is not list:
        stock_names = [stock_names]
    prices_df = loss_profit.prepare_adj_close(stock_names)   
    capitals =[]
    record_shares_df.date = record_buy_sell_df.Dates
    for day in record_shares_df.date:
        capital = record_buy_sell_df.Capitals.loc[record_buy_sell_df.Dates == day].values[0]
        for stock_name in stock_names:
            stock_amount = get_amount_by_name_and_date(record_shares_df, stock_name, day)
            if stock_amount > 0:
                price = prices_df['Adj_Close'].loc[np.logical_and((prices_df.Date == day),(prices_df.Name == stock_name))].values[0]
                capital += price*stock_amount
        capitals.append(capital)
    return pd.DataFrame.from_dict({'capital':capitals,'date':record_shares_df.date.copy()})

In [21]:
# create main money flow dataframe
# name : name of ETF
# our : final capital using our algorithm
# bah : final capital using buy and hold
# our_r: our annualized % return
# bah_r: bah annualized % return
# ant : annualized number of transaction
# pos : percent of success : sum(succ. transaction) / sum(transaction)
# apt : average percent profit per transactions
# l   : average transaction length
# mpt : maximum profit percentage in transaction
# mlt : maximum loss percentage in transaction
# maxc : maximum capital over test period
# minc : minimum capital over test period
money_flow_df = pd.DataFrame(columns=['name','our','bah','our_r','bah_r','ant','pos','apt','l','mpt','mlt','maxc','minc'])



In [22]:
import math
def day_between(date1_str,date2_str):
    return datetime.strptime(date1_str, '%Y-%m-%d') - datetime.strptime(date2_str, '%Y-%m-%d')

# name : name of ETF
def get_name(stock_name):
    return stock_name

# our : final capital using our algorithm
def get_our(final_capital):
    return final_capital

# bah : final capital using buy and hold
def get_bah(buy_and_hold_capital):
    return buy_and_hold_capital

# our_r: our annualized % return average
def get_our_r(transactions_df, period_of_days):
#     P : initial capital
#     n   : period in year(for day its 365)
#     t   : num of period observed: 1 means 365 day
#     A : final capital
    
    A = transactions_df.capital_after.iloc[-1]
    P = transactions_df.capital_before.iloc[0]
    n = 365
    t = period_of_days / n # our test period / one year
    r = n*((A/P)**(1/(n*t))-1)
    
    return 100*r

# bah_r: bah annualized % return
def get_bah_r(buy_and_hold_capital, period_of_days):
    A = buy_and_hold_capital
    P = 10000
    n = 365
    t = period_of_days / n # our test period / one year
    r = n*((A/P)**(1/(n*t))-1)
    
    return 100*r

# ant : annualized number of transaction
def get_ant(transactions_df,period_of_days):
    return transactions_df.shape[0] * 365 / period_of_days

# pos : percent of success : sum(succ. transaction) / sum(transaction)
def get_pos(transactions_df):
    pos_len = transactions_df.loc[transactions_df['return'] > 0].shape[0]
    return 100*pos_len / transactions_df.shape[0]

# apt : average percent profit per transactions
def get_apt(transactions_df):
    return 100*transactions_df['return'].sum() / transactions_df.shape[0]

# l   : average transaction length
def get_l(transactions_df):
    return transactions_df.t_period.sum() / transactions_df.shape[0]

# mpt : maximum profit percentage in transaction
def get_mpt(transactions_df):
    return 100*transactions_df['return'].max()

# mlt : maximum loss percentage in transaction
def get_mlt(transactions_df):
    return transactions_df['return'].min()

# maxc : maximum capital over test period
def get_maxc(transactions_df):
    return transactions_df.capital_after.max()

# minc : minimum capital over test period
def get_minc(transactions_df):
    return transactions_df.capital_after.min()


# # Unit Test
# # name : name of ETF
# get_name('spy')
# # our : final capital using our algorithm
# get_our(t_df)
# # bah : final capital using buy and hold
# get_bah('spy')
# # our_r: our annualized % return
# get_our_r(t_df)
# # bah_r: bah annualized % return
# get_bah_r('spy',420)
# # ant : annualized number of transaction
# get_ant(t_df,420)
# # pos : percent of success : sum(succ. transaction) / sum(transaction)
# get_pos(t_df)
# # apt : average percent profit per transactions
# get_apt(t_df)
# # l   : average transaction length
# get_l(t_df)
# # mpt : maximum profit percentage in transaction
# get_mpt(t_df)
# # mlt : maximum loss percentage in transaction
# get_mlt(t_df)
# # maxc : maximum capital over test period
# get_maxc(t_df)
# # minc : minimum capital over test period
# get_minc(t_df)

In [23]:
def create_money_flow_dataframe(stock_names, period_of_days = 420 * 365 / 250):
    col_order = ['name','our','bah','our_r','bah_r','ant','pos','apt','l','mpt','mlt','maxc','minc']

    money_flow_dict = {'name':[],
                       'our'  :[],
                       'bah'  :[],
                       'our_r':[],
                       'bah_r':[],
                       'ant'  :[],
                       'pos'  :[],
                       'apt'  :[],
                       'l'    :[],
                       'mpt'  :[],
                       'mlt'  :[],
                       'maxc' :[],
                       'minc' :[]                  
                      }
    for stock_name in stock_names:
           
        record_buy_sell_df, record_shares_df = make_transactions(stock_name)
        t_df = create_transaction_dataframe(record_buy_sell_df,record_shares_df, stock_name)
        final_capital = t_df.capital_after.iloc[-1]
        bah_capital = buy_and_hold_transaction(stock_name)[0]
        if type(stock_name) is list:
            print('ALL')
            money_flow_dict['name'].append(get_name('ALL'))
        else:
            print(stock_name+',', end=' ')    
            money_flow_dict['name'].append(get_name(stock_name))
            
        money_flow_dict['our'].append(get_our(final_capital))
        money_flow_dict['bah'].append(get_bah(bah_capital))
        money_flow_dict['our_r'].append(get_our_r(t_df,period_of_days))
        money_flow_dict['bah_r'].append(get_bah_r(bah_capital, period_of_days))
        money_flow_dict['ant'].append(get_ant(t_df, period_of_days))
        money_flow_dict['pos'].append(get_pos(t_df))
        money_flow_dict['apt'].append(get_apt(t_df))
        money_flow_dict['l'].append(get_l(t_df))
        money_flow_dict['mpt'].append(get_mpt(t_df))
        money_flow_dict['mlt'].append(get_mlt(t_df))
        money_flow_dict['maxc'].append(get_maxc(t_df))
        money_flow_dict['minc'].append(get_minc(t_df))

    return pd.DataFrame.from_dict(money_flow_dict)[col_order]

    

In [24]:
stock_names = [['spy', 'xlf', 'xlu', 'xle','xlp', # this second list means take all shares and conduct our CTA
                'xli', 'xlv', 'xlk', 'ewj',
                'xlb', 'xly', 'eww', 'dia',
                'ewg', 'ewh', 'ewc', 'ewa'],
    
                    'spy', 'xlf', 'xlu', 'xle',
                   'xlp', 'xli', 'xlv', 'xlk', 'ewj',
                   'xlb', 'xly', 'eww', 'dia', 'ewg',
                   'ewh', 'ewc', 'ewa'
               
              ]
period_of_days = 420 * 365 / 250 #tfixed for working days test sample for each
# stock_names = ['spy']

m_flow_df = create_money_flow_dataframe(stock_names)
m_flow_df.ix[:,1:] = m_flow_df.ix[:,1:].applymap(lambda x: "{0:.2f}".format(x))
m_flow_df.to_csv("../result/money_flow_data.csv",index=False)

ALL
spy, xlf, xlu, xle, xlp, xli, xlv, xlk, ewj, xlb, xly, eww, dia, ewg, ewh, ewc, ewa, 

C:\Anaconda3\lib\site-packages\ipykernel_launcher.py:16: DeprecationWarning: 
.ix is deprecated. Please use
.loc for label based indexing or
.iloc for positional indexing

See the documentation here:
http://pandas.pydata.org/pandas-docs/stable/indexing.html#deprecate_ix
  app.launch_new_instance()


In [25]:
m_flow_df

,name,our,bah,our_r,bah_r,ant,pos,apt,l,mpt,mlt,maxc,minc
0,ALL,54111.13,11046.03,100.67,5.92,94.64,79.87,2.90,8.24,54.21,-0.04,54111.13,10119.69
1,spy,8549.28,11951.91,-9.30,10.61,52.38,39.77,-0.12,3.44,3.16,-0.05,10287.12,8431.75
2,xlf,30541.15,10561.70,66.55,3.25,57.14,79.17,1.22,3.43,10.85,-0.05,30541.15,10065.06
3,xlu,30206.49,11921.92,65.89,10.47,51.19,84.88,1.33,3.95,5.91,-0.01,30206.49,10114.85
4,xle,46492.65,8999.16,91.62,-6.28,61.90,73.08,1.55,2.94,15.24,-0.03,46492.65,10119.69
5,xlp,21508.94,11082.40,45.65,6.12,60.12,83.17,0.80,3.03,4.92,-0.04,21508.94,9996.15
6,xli,23959.47,12538.18,52.08,13.47,61.31,72.82,0.90,2.94,8.19,-0.04,24016.39,9963.00
7,xlv,27594.79,10973.35,60.50,5.53,59.52,79.00,1.06,3.40,8.36,-0.04,27644.09,9916.56
8,xlk,27835.00,13830.50,61.02,19.31,60.12,82.18,1.06,3.22,7.65,-0.04,27835.00,9997.23
9,ewj,32804.14,10841.96,70.81,4.81,59.52,84.00,1.24,3.42,8.26,-0.04,32804.14,9871.80


In [26]:
A = s_t_df.capital_after.iloc[-1]
P = s_t_df.capital_before.iloc[0]
n = 365
t = 1.68
r = n*((A/P)**(1/(n*t))-1)
s_t_df

NameError: name 's_t_df' is not defined

In [ ]:
r